# 06 · Agente GenAI — Asesor de Estrategia Comercial
## LangChain LCEL + Anthropic · Insights personalizados por segmento

**Objetivo:**

Dado un vendedor y su segmento asignado, genera un diagnóstico y una estrategia comercial
personalizada, orientada al perfil del cluster al que pertenece.

**Datos de entrada por vendedor:**
- Categorías principales del catálogo
- Stock disponible
- Precio Promedio
- Reputación

**Flujo de la cadena LCEL:**
```
cluster_profiles.md  ┐
datos del vendedor   ┼──► ChatPromptTemplate ──► ChatAnthropic ──► StrOutputParser
títulos marketplace  ┘                                              ↓
                                                          estrategia accionable
```


In [1]:
import os
from pathlib import Path

# Garantiza que el CWD sea la raíz del repositorio 
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)


## Importaciones y variables de entorno

In [2]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

from seller_segmentation.data.loader import load_df
from seller_segmentation.features.builder import construir_perfil_seller_agente
from seller_segmentation.models.clustering import SellerClusterer
from seller_segmentation.models.genai_extension import (
    CLUSTER_NAMES,
    StrategyAdvisor,
    _SYSTEM_PROMPT,
    _HUMAN_TEMPLATE,
)

load_dotenv()
assert os.getenv("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY no configurada – revisar el archivo .env"
print("API key cargada correctamente.")

API key cargada correctamente.


## 1. Datos de los seller

Con base en el diseño del agente generativo — cuyo propósito es producir estrategias comerciales personalizadas según el perfil del clúster al que pertenece cada vendedor —, se definen los siguientes atributos de entrada: seller_nickname, stock, price, categoria, reputacion, cluster y cluster_name.

Para construir el dataset de entrada, se parte de los datos preprocesados y los datos crudos. Dado que un mismo vendedor puede estar asociado a múltiples categorías, se aplica un tratamiento de agregación que consiste en generar un campo concatenado con las 5 categorías más representativas del vendedor, ordenadas por stock y precio. Este campo sintetiza el perfil categórico del vendedor en un formato compacto y consumible por el agente.

In [3]:
# Cargar datos crudos y los labels
df_raw = load_df()
print(f"Dataset crudo cargado: {df_raw.shape}")
labels = labels = pd.read_csv("data/processed/seller_labels.csv", index_col=0)[['seller_nickname','cluster']]

Dataset crudo cargado: (185250, 14)


In [4]:
df_clusters = labels

# Construir perfil a nivel seller para el agente
df_perfil_sellers = construir_perfil_seller_agente(
    df_productos=df_raw,
    df_clusters=df_clusters,
    col_seller="seller_nickname",
    col_precio="price",
    col_stock="stock",
    col_categoria="category_name",
    col_reputacion="seller_reputation",
    col_cluster="cluster",
    top_n_categorias=5,
)

print(f"\nPerfil de sellers construido: {df_perfil_sellers.shape}")
df_perfil_sellers.head(5)


Perfil de sellers construido: (46586, 7)


,seller_nickname,stock,price,categorias_seller,reputacion,cluster,cluster_name
0,000631669c,10,799.0,OTROS,newbie,0,Exploradores
1,0007153bca,55,399.0,OTROS,green,0,Exploradores
2,000bee3c3b,0,382.5,"LIBROS, MULTIMEDIA Y OTROS",newbie,0,Exploradores
3,000df2bd02,5,1550.0,ACCESORIOS PARA AUTOS Y CAMIONETAS,green,0,Exploradores
4,000e27cea2,6,457.5,"ARTÍCULOS DEL HOGAR, SALUD",green_silver,0,Exploradores


In [5]:
# Seleccionar un vendedor de ejemplo que tenga cluster asignado
df_ejemplo = df_perfil_sellers.dropna(subset=["cluster"]).iloc[15]

example_seller  = df_ejemplo["seller_nickname"]
example_cluster = int(df_ejemplo["cluster"])
example_metrics = df_perfil_sellers.set_index("seller_nickname").loc[example_seller]

print(f"Vendedor: {example_seller}  |  Cluster: {example_cluster} – {CLUSTER_NAMES[example_cluster]}")
print("\n--- Métricas clave ---")
print(example_metrics.to_string())

Vendedor: 0021be2b0e  |  Cluster: 2 – Consolidados

--- Métricas clave ---
stock                        1774
price                       235.0
categorias_seller        TEXTILES
reputacion           green_silver
cluster                         2
cluster_name         Consolidados


## 2· Perfiles de clusters

Los perfiles de cada segmento se leen desde `data/cluster_profiles.md`.  (`cluster_profiles.md`)

Este archivo se inyecta íntegramente en el prompt, lo que permite:
- Contextualizar al LLM sobre las características de cada segmento
- Orientar la estrategia según el arquetipo del cluster
- Actualizar los perfiles sin modificar el código

In [6]:
PROFILES_PATH = Path("data/cluster_profiles.md")

cluster_profiles_text = PROFILES_PATH.read_text(encoding="utf-8")
print(cluster_profiles_text)

# Perfiles de Segmentos — Sellers MercadoLibre

Segmentación obtenida con KMeans (k=4, Silhouette Score=0.58) sobre 46.586 vendedores.
Los cuatro segmentos representan arquetipos con lógicas comerciales, logísticas y de precios
fundamentalmente distintas.

---

## Cluster 0 — Exploradores (75.1 % de los sellers)

**Perfil:** Vendedor individual que usa la plataforma como canal secundario o de ingreso
complementario. Vende tanto nuevo como usado, opera con catálogo pequeño concentrado en pocas
categorías y mantiene precios fijos sin estrategia de variación. Su reputación es baja
(nivel amarillo) y su logística básica (principalmente XD). Es la base masiva del ecosistema.

**Desafío principal:** Generar ventas consistentes, construir reputación positiva y establecer
una operación estable en la plataforma.

**Accionables:**
- Responder consultas y reclamos en menos de 24 horas para acumular calificaciones positivas
  y subir de nivel de reputación amarillo a verde en el menor tiempo posib


## 3. Prompt Template

El template tiene dos partes:

| Mensaje    | Rol       | Propósito                                               |
|------------|-----------|---------------------------------------------------------|
| `system`   | Contexto  | Define el rol del agente y el tono de respuesta         |
| `human`    | Instrucción | Inyecta los datos del vendedor y los perfiles de cluster |

Los **slots del template** se rellenan dinámicamente en cada invocación:

| Slot                 | Fuente de datos                                      |
|----------------------|------------------------------------------------------|
| `{cluster_profiles}` | `cluster_profiles.md` completo                   |
| `{seller_id}`        | ID / Nickname del vendedor                           |
| `{cluster}`          | Etiqueta numérica del cluster                        |
| `{cluster_name}`     | Nombre estratégico del segmento                      |
| `{categories}`       | Top 5 categorías por valor de inventario             |
| `{stock}`            | Stock total disponible                               |
| `{price}`            | Precio promedio del catálogo                         |
| `{reputation}`       | Reputación ordinal del seller                        |


In [7]:
# --- Prompt de sistema ---
print("=" * 60)
print("SYSTEM PROMPT")
print("=" * 60)
print(_SYSTEM_PROMPT)

SYSTEM PROMPT
Eres un especialista en estrategia comercial con experiencia en marketplaces. Tu rol es
acompañar a vendedores de MercadoLibre en el diseño de estrategias de crecimiento adaptadas
a su perfil y segmento. Tu análisis parte siempre de los datos reales del vendedor: reconoces
sus fortalezas actuales y las usas como punto de partida para proponer iniciativas de alto
impacto.

Tu tono es profesional, directo y orientado al negocio. Escribe en español, en segunda persona
singular al dirigirte al vendedor. Cada recomendación debe ser específica, accionable y
justificada con base en los datos disponibles. Usa viñetas y no superes las 300 palabras.


In [8]:
# --- Template del mensaje humano ---
print("=" * 60)
print("HUMAN TEMPLATE (slots sin rellenar)")
print("=" * 60)
print(_HUMAN_TEMPLATE)

HUMAN TEMPLATE (slots sin rellenar)
## Perfiles de segmentos de vendedores en MercadoLibre

{cluster_profiles}

---

## Datos del vendedor a analizar

- **Seller_Nickname:** {seller_id}
- **Segmento:** {cluster_name} (Cluster {cluster})
- **Categorías principales:** {categories}
- **Stock total disponible:** {stock}
- **Precio promedio:** {price}
- **Reputación:** {reputation}

---

Con base en el perfil del segmento y los datos del vendedor, elabora una ficha de estrategia
comercial personalizada con las siguientes secciones:

1. **Diagnóstico del negocio** — identifica 1 ventaja competitiva real del vendedor que puede
   aprovecharse como base para crecer dentro de su segmento.
2. **Estrategia de crecimiento** — propón 2 iniciativas comerciales concretas, ordenadas por
   prioridad, que el vendedor pueda implementar para aumentar sus ventas en el corto plazo.
3. **Indicador clave de éxito** — define el métrca principal que permitirá evaluar el impacto
   de la estrategia y hacer segu

In [9]:
# Construir el objeto ChatPromptTemplate
prompt_template = ChatPromptTemplate.from_messages([
    ("system", _SYSTEM_PROMPT),
    ("human", _HUMAN_TEMPLATE),
])

print("Variables del template:", prompt_template.input_variables)

Variables del template: ['categories', 'cluster', 'cluster_name', 'cluster_profiles', 'price', 'reputation', 'seller_id', 'stock']


## 4. Construcción de la cadena LCEL

La **cadena LCEL** (LangChain Expression Language) conecta tres componentes con el operador `|`:

```
prompt_template  |  ChatAnthropic(model)  |  StrOutputParser()
```

- **`prompt_template`** — formatea el dict de entradas en mensajes listos para el LLM
- **`ChatAnthropic`** — invoca la API de Anthropic con el modelo configurado
- **`StrOutputParser`** — extrae el texto plano de la respuesta del modelo

In [10]:
# Componentes individuales
llm    = ChatAnthropic(model="claude-sonnet-4-6", max_tokens=1024)
parser = StrOutputParser()

# Cadena LCEL
chain = prompt_template | llm | parser

print("Cadena construida:", chain)
print("\nPasos de la cadena:")
for step in chain.steps:
    print(f"  • {type(step).__name__}")

Cadena construida: first=ChatPromptTemplate(input_variables=['categories', 'cluster', 'cluster_name', 'cluster_profiles', 'price', 'reputation', 'seller_id', 'stock'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='Eres un especialista en estrategia comercial con experiencia en marketplaces. Tu rol es\nacompañar a vendedores de MercadoLibre en el diseño de estrategias de crecimiento adaptadas\na su perfil y segmento. Tu análisis parte siempre de los datos reales del vendedor: reconoces\nsus fortalezas actuales y las usas como punto de partida para proponer iniciativas de alto\nimpacto.\n\nTu tono es profesional, directo y orientado al negocio. Escribe en español, en segunda persona\nsingular al dirigirte al vendedor. Cada recomendación debe ser específica, accionable y\njustificada con base en los datos disponibles. Usa viñetas y no superes las 300 palabras.'), additio

In [11]:
# Vista previa del prompt renderizado (sin llamar al LLM)
from seller_segmentation.models.genai_extension import _format_reputation

sample_input = {
    "cluster_profiles": cluster_profiles_text,
    "seller_id":     example_seller,
    "cluster":       example_cluster,
    "cluster_name":  CLUSTER_NAMES.get(example_cluster, f"Cluster {example_cluster}"),
    "categories":    str(example_metrics.get("categorias_seller", "No disponible")),
    "stock":         f"{float(example_metrics.get('stock', 0)):.0f} unidades",
    "price":         f"${float(example_metrics.get('price', 0)):,.2f}",
    "reputation":    _format_reputation(str(example_metrics.get("reputacion", ""))),
}

rendered = prompt_template.invoke(sample_input)
print("--- PROMPT RENDERIZADO ---")
for msg in rendered.messages:
    print(f"[{msg.type.upper()}]")
    content = msg.content if isinstance(msg.content, str) else str(msg.content)
    print(content[:800], "..." if len(content) > 1000 else "")


--- PROMPT RENDERIZADO ---
[SYSTEM]
Eres un especialista en estrategia comercial con experiencia en marketplaces. Tu rol es
acompañar a vendedores de MercadoLibre en el diseño de estrategias de crecimiento adaptadas
a su perfil y segmento. Tu análisis parte siempre de los datos reales del vendedor: reconoces
sus fortalezas actuales y las usas como punto de partida para proponer iniciativas de alto
impacto.

Tu tono es profesional, directo y orientado al negocio. Escribe en español, en segunda persona
singular al dirigirte al vendedor. Cada recomendación debe ser específica, accionable y
justificada con base en los datos disponibles. Usa viñetas y no superes las 300 palabras. 
[HUMAN]
## Perfiles de segmentos de vendedores en MercadoLibre

# Perfiles de Segmentos — Sellers MercadoLibre

Segmentación obtenida con KMeans (k=4, Silhouette Score=0.58) sobre 46.586 vendedores.
Los cuatro segmentos representan arquetipos con lógicas comerciales, logísticas y de precios
fundamentalmente distin

## 5. Estrategia para un vendedor individual

Se usa `StrategyAdvisor`, que encapsula la cadena y el formateo de datos.
El parámetro `titles` recibe la lista de títulos extraída del dataset crudo.

In [12]:
advisor = StrategyAdvisor(profiles_path=PROFILES_PATH)

In [13]:
strategy = advisor.advise(
    seller_id=example_seller,
    cluster=example_cluster,
    metrics=example_metrics,
)

print(f"Vendedor : {example_seller}")
print(f"Segmento : Cluster {example_cluster} – {CLUSTER_NAMES[example_cluster]}")
print("=" * 60)
print(strategy)


Vendedor : 0021be2b0e
Segmento : Cluster 2 – Consolidados
# Ficha de Estrategia Comercial — Vendedor 0021be2b0e

---

## 1. Diagnóstico del negocio

**Ventaja competitiva identificada: Reputación Plata en un segmento de gran volumen.**

Dentro de los Consolidados, donde el volumen masivo suele presionar a la baja la calidad de servicio, alcanzar nivel Plata (3/8) es una señal de operación consistente. En textiles —categoría con alta competencia y márgenes ajustados— esa reputación te diferencia de operadores similares y genera confianza en compradores que comparan múltiples opciones. Es tu activo más defendible a corto plazo y la base desde la cual escalar visibilidad.

---

## 2. Estrategia de crecimiento

**Iniciativa 1 (Prioridad alta): Activar Product Ads en las publicaciones de mayor margen.**

Con 1.774 unidades disponibles y un precio promedio de $235, tu catálogo de textiles compite en un segmento donde la posición en búsqueda define el volumen. Identificá tus 10 a 15 publicaci

In [14]:
# Seleccionar otro vendedor de ejemplo que tenga cluster asignado
df_ejemplo2 = df_perfil_sellers.dropna(subset=["cluster"]).iloc[45000]

example_seller2  = df_ejemplo2["seller_nickname"]
example_cluster2 = int(df_ejemplo2["cluster"])
example_metrics2 = df_perfil_sellers.set_index("seller_nickname").loc[example_seller2]

print(f"Vendedor: {example_seller2}  |  Cluster: {example_cluster2} – {CLUSTER_NAMES[example_cluster2]}")
print("\n--- Métricas clave ---")
print(example_metrics.to_string())

Vendedor: f71aa53819  |  Cluster: 0 – Exploradores

--- Métricas clave ---
stock                        1774
price                       235.0
categorias_seller        TEXTILES
reputacion           green_silver
cluster                         2
cluster_name         Consolidados


In [15]:
strategy2 = advisor.advise(
    seller_id=example_seller2,
    cluster=example_cluster2,
    metrics=example_metrics2,
)

print(f"Vendedor : {example_seller2}")
print(f"Segmento : Cluster {example_cluster2} – {CLUSTER_NAMES[example_cluster2]}")
print("=" * 60)
print(strategy2)

Vendedor : f71aa53819
Segmento : Cluster 0 – Exploradores
# Ficha de Estrategia Comercial — f71aa53819

---

## 1. Diagnóstico del negocio

Estás clasificado en el segmento de Exploradores, pero tu reputación Platinum te posiciona radicalmente por encima de tus pares: mientras el 75% de los vendedores en este cluster opera con nivel amarillo, tú ya alcanzaste el nivel más alto del ecosistema. Esa credibilidad acumulada es tu activo diferencial más valioso y el punto de partida para cualquier estrategia de crecimiento.

El desafío real no es reputación, sino capitalización: con 149 unidades de stock y un precio promedio de $841, tienes capacidad operativa activa que todavía no está rindiendo a la altura de tu nivel de confianza en la plataforma.

---

## 2. Estrategia de crecimiento

**Iniciativa 1 — Relanzar el catálogo con foco en visibilidad (prioridad alta)**
Tu reputación Platinum mejora el posicionamiento orgánico de tus publicaciones, pero ese beneficio solo se activa si los list

In [17]:
# Seleccionar otro vendedor de ejemplo que tenga cluster asignado
df_ejemplo3 = df_perfil_sellers.dropna(subset=["cluster"]).iloc[46552]

example_seller3  = df_ejemplo3["seller_nickname"]
example_cluster3 = int(df_ejemplo3["cluster"])
example_metrics3 = df_perfil_sellers.set_index("seller_nickname").loc[example_seller3]

print(f"Vendedor: {example_seller3}  |  Cluster: {example_cluster3} – {CLUSTER_NAMES[example_cluster3]}")
print("\n--- Métricas clave ---")
print(example_metrics.to_string())

Vendedor: ffd38f3288  |  Cluster: 1 – Exclusivos

--- Métricas clave ---
stock                        1774
price                       235.0
categorias_seller        TEXTILES
reputacion           green_silver
cluster                         2
cluster_name         Consolidados


In [18]:
strategy3 = advisor.advise(
    seller_id=example_seller3,
    cluster=example_cluster3,
    metrics=example_metrics3,
)

print(f"Vendedor : {example_seller3}")
print(f"Segmento : Cluster {example_cluster3} – {CLUSTER_NAMES[example_cluster3]}")
print("=" * 60)
print(strategy3)

Vendedor : ffd38f3288
Segmento : Cluster 1 – Exclusivos
# Ficha de Estrategia Comercial — ffd38f3288

**Segmento:** Exclusivos · **Reputación:** Platinum · **Precio promedio:** $1.531,49

---

## 1. Diagnóstico del negocio

**Ventaja competitiva principal: reputación Platinum con catálogo de nicho.**

Operar en el nivel más alto de reputación dentro del segmento Exclusivos te posiciona muy por encima del promedio del ecosistema. Con un precio promedio de $1.531,49 y categorías como Fragancias, Maquillaje y Calzado —donde el comprador valora fuertemente la confianza antes de comprar—, tu reputación actúa como un diferenciador real que reduce la fricción en la decisión de compra. Ese activo está siendo subutilizado con solo 21 unidades en stock.

---

## 2. Estrategia de crecimiento

**Iniciativa 1 (alta prioridad): Ampliar catálogo dentro de las categorías actuales**

Tu reputación Platinum justifica que los compradores elijan tus publicaciones incluso compitiendo en precio. Sin embargo

In [19]:
# Seleccionar otro vendedor de ejemplo que tenga cluster asignado
df_ejemplo4 = df_perfil_sellers.dropna(subset=["cluster"]).iloc[1074]

example_seller4  = df_ejemplo4["seller_nickname"]
example_cluster4 = int(df_ejemplo4["cluster"])
example_metrics4 = df_perfil_sellers.set_index("seller_nickname").loc[example_seller4]

print(f"Vendedor: {example_seller4}  |  Cluster: {example_cluster4} – {CLUSTER_NAMES[example_cluster4]}")
print("\n--- Métricas clave ---")
print(example_metrics.to_string())


Vendedor: 0605ee1983  |  Cluster: 3 – Prometedores

--- Métricas clave ---
stock                        1774
price                       235.0
categorias_seller        TEXTILES
reputacion           green_silver
cluster                         2
cluster_name         Consolidados


In [20]:
strategy4 = advisor.advise(
    seller_id=example_seller4,
    cluster=example_cluster4,
    metrics=example_metrics4,
)

print(f"Vendedor : {example_seller4}")
print(f"Segmento : Cluster {example_cluster4} – {CLUSTER_NAMES[example_cluster4]}")
print("=" * 60)
print(strategy4)

Vendedor : 0605ee1983
Segmento : Cluster 3 – Prometedores
# Ficha de Estrategia Comercial — Vendedor 0605ee1983

---

## 1. Diagnóstico del negocio

**Ventaja competitiva:** Tu reputación Platinum (nivel 1/8) es tu activo más valioso y diferenciador. En categorías de alta competencia como Teléfonos Inteligentes y Cámaras, la reputación es un factor de desempate directo en el algoritmo de búsqueda de MercadoLibre. La mayoría de tus competidores directos no operan desde ese nivel, lo que te da una ventaja de visibilidad orgánica que aún puedes explotar más agresivamente.

**Señal de alerta:** Con solo 83 unidades de stock y un precio promedio de $1.136, tu inventario disponible es reducido para un vendedor Prometedor. Esto limita la escala y puede generar cancelaciones si la demanda se activa sin reposición oportuna.

---

## 2. Estrategia de crecimiento

**Iniciativa 1 — Concentrar stock en los productos de mayor conversión (prioridad alta)**
Identifica cuáles de tus 83 unidades tienen 

## Conclusiones

- Validación de hipótesis

| Hipótesis | Resultado | Evidencia |
|-----------|-----------|-----------|
| H1: Los sellers difieren significativamente en tamaño de catálogo y distribución de precios | Confirmada | Los sellers difieren significativamente en tamaño de catálogo y distribución de precios. stock_mean varía de 19 (Explorador) a 328 (Consolidado) — 17x de diferencia. price_max varía de 1.736 a 5.409 entre segmentos. |
| H2: Un grupo reducido de sellers concentra una proporción desproporcionada del volumen  | Confirmada | Un grupo reducido de sellers concentra una proporción desproporcionada del volumen. El 23.3% de sellers (Consolidados + Prometedores) concentra el 66.3% del stock y genera ingresos proxy 4–14x superiores al C2C. |
| H3: La estrategia de precios (premium vs. descuento) correlaciona con la amplitud del catálogo| Confirmada parcialmente | La hipótesis se confirma para C0, C1 y C3: sellers con catálogos concentrados tienen precios más uniformes, y sellers con catálogos más diversos muestran mayor amplitud de precios. C2 (Mayorista) es la excepción: catálogo amplio pero precios homogéneos por estrategia de volumen/descuento masivo. |
| H4: Existen arquetipos de sellers diferenciados y recuperables mediante clustering  | Confirmada | Criterio de éxito: Silhouette ≥ 0.35. Silhouette Score obtenido: 0.58 → supera el umbral en 66%. Los 4 arquetipos son interpretables comercialmente y traducibles a acciones concretas. |

<br>

- Conclusiones del modelo

   - Técnicas:
      - KMeans con k=4 produce una segmentación estadísticamente sólida: Silhouette=0.58, Calinski-Harabasz=33.694, Davies-Bouldin=0.71 — consistentes entre sí y por encima de umbrales aceptables
      - El método del codo identifica k=4 de forma estable y reproducible con random_state=42
      - El cluster C2C (76.7% del universo) presenta mayor dispersión interna, esperado dado su tamaño y heterogeneidad; los tres segmentos minoritarios están bien definidos y son directamente accionables

   - De negocio:

|Segmento|% Sellers|% Stock|Ingreso proxy promedio|Palanca principal|
|-----|------|------|-----|------|
|Explorador (C0)|75.1%|23.0%|15.590|Activación y retención|
|Exclusivo (C1)|7.1%|10.6%|45.840|Confianza y expansión de catálogo|
|Consolidado (C2)|9.1%|43.1%|205.897|Rotación de inventario y visibilidad|
|Prometedor (C3)|8.7%|23.2%|89.404|Escala logística y diversificación|

   El ecosistema sigue una estructura de marketplace maduro: una base masiva de cola larga y una minoría que mueve la mayor parte del valor.

<br>

- Preguntas Planteadas inicialmente

   1. ¿Cuántos segmentos? → 4, estadísticamente validados y comercialmente interpretables
   2. ¿Qué diferencia al seller de alto valor? → Logística profesional (FBM/DS), precio alto o volumen masivo, y reputación ordinal ≤ 4 (green/silver). El stock_mean y la amplitud de precios son los discriminadores más fuertes
   3. ¿A quién contactar primero? → Consolidados (máximo ingreso proxy, palanca de rotación) y Prometedores (mayor potencial de crecimiento, infraestructura ya lista)
   4. ¿Cómo asignar un nuevo seller? → El modelo entrenado (seller_clusterer.joblib) permite predict() en tiempo real sobre nuevos sellers escalados con el mismo RobustScaler

<br>

- Siguientes pasos

   - Corto plazo:
      - Complementar asesor GenAI con un RAG que contenga información de políticas de seguridad, tratamiento de datos y verbosity Mercado Libre.
      - Evaluar sub-segmentación del Cluster 0 (Explorador) — por su heterogeneidad interna podría justificar k=5 o k=6 en una segunda iteración
      - Robustecer el perfilamiento utilizando otras variables, como variables financieras de los sellers, o variables de temporalidad para conocer el comportamiento histórico de los sellers.

   - Mediano plazo (producto):

      - Conectar el modelo a un pipeline de scoring en batch (mensual) que reasigne clusters a medida que los sellers evolucionan
      - Diseñar un dashboard para el equipo comercial con distribución por cluster, ingreso proxy y alertas de movimiento entre segmentos
      - Ampliar el asesor GenAI (notebook 06) para que genere estrategias comparativas entre el estado actual y el cluster objetivo al que el seller podría ascender
